# 01 — Exploring the DRB Stream Temperature Data

This notebook walks through the structure of the `.npz` data files provided for the course project.  
Each file corresponds to one USGS stream temperature monitoring site in the **Delaware River Basin (DRB)**.

**Before running this notebook**, make sure you have:
1. Downloaded the `.npz` files from Google Drive (link in README.md)
2. Placed them in the `data/` folder in this directory

The five sites are:

| Site ID | Associated Reservoir(s) |
|---------|------------------------|
| 1573    | Cannonsville + Pepacton |
| 1571    | Cannonsville |
| 1565    | Cannonsville |
| 1450    | Pepacton |
| 1641    | Neversink |

In [7]:
import numpy as np
import pandas as pd

DATA_DIR = '../../public/eacvi-fhnn-data/site_wise_data_static_infilled/sequence_length_360'
SITE_IDS = [1573, 1571, 1565, 1450, 1641]

# 15 input features — col 14 (stream_temp) is the ar1 feature, zeroed at forecast steps
FEATURE_NAMES = [
    'tmin', 'tmax', 'srad', 'pr', 'vs', 'rmax', 'rmin', 'rmean',
    'spillway', 'releases',
    'seg_slope', 'seg_elev', 'seg_width', 'seg_length',
    'stream_temp'
]

## 1. NPZ file structure

Load one site and inspect all keys.

In [8]:
site = 1573
data = np.load(f'{DATA_DIR}/{site}.npz', allow_pickle=True)

print(f'Keys in {site}.npz:')
for key in data.files:
    val = data[key]
    if hasattr(val, 'shape'):
        print(f'  {key:<30s}  shape={val.shape}  dtype={val.dtype}')
    else:
        print(f'  {key:<30s}  value={val}')

Keys in 1573.npz:
  input_shape                     shape=(2,)  dtype=int64
  pretrain_X                      shape=(12754, 15)  dtype=float64
  pretrain_Y                      shape=(12754,)  dtype=float64
  finetune_X                      shape=(13132, 15)  dtype=float64
  finetune_Y                      shape=(13132,)  dtype=float64
  finetune_Y_mask                 shape=(13132,)  dtype=bool
  forecast_E0_X                   shape=(708, 15)  dtype=float64
  forecast_E0_Y                   shape=(708,)  dtype=float64
  forecast_infill_mask            shape=(708,)  dtype=bool
  forecast_dates                  shape=(708,)  dtype=datetime64[ns]
  forecast_pb_temp                shape=(708,)  dtype=float64
  forecast_E1_X                   shape=(708, 15)  dtype=float64
  forecast_E1_Y                   shape=(708,)  dtype=float64
  forecast_E2_X                   shape=(708, 15)  dtype=float64
  forecast_E2_Y                   shape=(708,)  dtype=float64
  forecast_E3_X               

### Key groups explained

| Group | Keys | Description |
|-------|------|-------------|
| **Pre-training** | `pretrain_X`, `pretrain_Y` | 1985-05-01 → 2020-04-01. Uses process-based (dwallin) model temperatures as targets — lots of data, useful for pre-training. |
| **Fine-tuning** | `finetune_X`, `finetune_Y`, `finetune_Y_mask` | 1985-05-01 → 2021-04-14. Mix of real observations and dwallin-infilled targets. `finetune_Y_mask=True` where the target is infilled (not a real thermometer reading). |
| **Forecast ensembles** | `forecast_E0_X` … `forecast_E30_X` and `forecast_E0_Y` … `forecast_E30_Y` | Forecast period (Apr–Sep 2021). 31 NOAA GEFS weather ensemble members, each giving a different plausible future weather trajectory. |
| **Forecast metadata** | `forecast_dates`, `forecast_infill_mask`, `forecast_pb_temp` | Dates, which rows used infilled context, and the fully process-based temperature column. |

## 2. Input features (X)

Each row of X is one day. There are **15 columns**.

In [9]:
X = data['finetune_X'].astype(np.float32)
Y = data['finetune_Y'].astype(np.float32)
mask = data['finetune_Y_mask'].astype(bool)

print(f'finetune_X shape : {X.shape}  ({X.shape[0]} days x {X.shape[1]} features)')
print(f'finetune_Y shape : {Y.shape}  (next-day stream temp, deg C)')
print()

finetune_X shape : (13132, 15)  (13132 days x 15 features)
finetune_Y shape : (13132,)  (next-day stream temp, deg C)



In [10]:
# Feature table with column index, name, and description
descriptions = [
    'Daily min air temperature (degC, GridMET)',
    'Daily max air temperature (degC, GridMET)',
    'Solar radiation (W/m2, GridMET)',
    'Precipitation (mm, GridMET)',
    'Wind speed (m/s, GridMET)',
    'Relative humidity max (%, GridMET)',
    'Relative humidity min (%, GridMET)',
    'Relative humidity mean (%, GridMET)',
    'Spillway release from site closest reservoir (cms)',
    'Controlled release from site closest reservoir (cms)',
    'Stream segment slope — static site attribute',
    'Segment elevation — static site attribute',
    'Segment width — static site attribute',
    'Segment length — static site attribute',
    'Previous day stream temp, ar1 (degC)  <-- MASKED at forecast steps',
]
feat_df = pd.DataFrame({'col_idx': range(15), 'name': FEATURE_NAMES, 'description': descriptions})
print(feat_df.to_string(index=False))

 col_idx        name                                                        description
       0        tmin                          Daily min air temperature (degC, GridMET)
       1        tmax                          Daily max air temperature (degC, GridMET)
       2        srad                                    Solar radiation (W/m2, GridMET)
       3          pr                                        Precipitation (mm, GridMET)
       4          vs                                          Wind speed (m/s, GridMET)
       5        rmax                                 Relative humidity max (%, GridMET)
       6        rmin                                 Relative humidity min (%, GridMET)
       7       rmean                                Relative humidity mean (%, GridMET)
       8    spillway                 Spillway release from site closest reservoir (cms)
       9    releases               Controlled release from site closest reservoir (cms)
      10   seg_slope            

## 3. Summary statistics

In [11]:
stats = pd.DataFrame(X, columns=FEATURE_NAMES).describe().T
pd.set_option('display.float_format', '{:.3f}'.format)
stats[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]

,count,mean,std,min,25%,50%,75%,max
tmin,13132.000,13.719,10.768,-19.670,4.301,14.788,23.342,34.621
tmax,13132.000,1.906,9.655,-29.005,-5.043,1.996,10.140,21.975
srad,13132.000,0.126,0.287,0.000,0.000,0.000,0.111,6.003
pr,13132.000,163.500,82.642,0.000,92.940,156.311,233.827,328.094
vs,13132.000,4.889,1.985,0.800,3.400,4.600,6.100,13.655
rmax,13132.000,88.911,11.112,28.681,81.925,91.309,100.000,100.000
rmin,13132.000,45.561,10.527,9.199,38.818,45.683,52.239,100.000
rmean,13132.000,67.236,9.493,19.162,61.639,68.537,74.048,100.000
spillway,13132.000,7.134,23.532,0.000,0.000,0.000,0.000,760.454
releases,13132.000,10.715,12.193,0.000,1.271,5.652,15.121,59.935


In [12]:
# Target variable summary
print('Target (next-day stream temperature, degC):')
print(pd.Series(Y, name='stream_temp_next_day').describe())

Target (next-day stream temperature, degC):
count   13132.000
mean       10.265
std         7.786
min        -0.100
25%         2.590
50%        10.000
75%        17.100
max        28.900
Name: stream_temp_next_day, dtype: float64


## 4. Observed vs infilled targets

`finetune_Y_mask=True` means the target for that day is a **dwallin process-based model prediction**, not a real thermometer reading.  
When you train your LSTM you may want to weight these differently or exclude them from the loss.

In [13]:
n_infilled = mask.sum()
n_observed = (~mask).sum()
print(f'Total finetune rows  : {len(mask)}')
print(f'  Observed (real)    : {n_observed}  ({100*n_observed/len(mask):.1f}%)')
print(f'  Infilled (dwallin) : {n_infilled}  ({100*n_infilled/len(mask):.1f}%)')

Total finetune rows  : 13132
  Observed (real)    : 9974  (76.0%)
  Infilled (dwallin) : 3158  (24.0%)


## 5. First few rows

In [14]:
df = pd.DataFrame(X[:5], columns=FEATURE_NAMES)
df['target_next_day'] = Y[:5]
df['target_is_infilled'] = mask[:5]
df.round(3)

,tmin,tmax,srad,pr,vs,rmax,rmin,rmean,spillway,releases,seg_slope,seg_elev,seg_width,seg_length,stream_temp,target_next_day,target_is_infilled
0,14.904,4.205,0.218,168.167,3.000,91.380,32.491,61.935,0.000,0.701,0.001,256.440,49.686,2511.610,6.310,7.320,True
1,16.575,0.440,0.234,224.126,5.300,97.686,33.230,65.458,0.000,0.701,0.001,256.440,49.686,2511.610,7.320,7.060,True
2,17.255,-1.995,0.000,299.430,4.400,63.405,23.474,43.440,0.000,0.701,0.001,256.440,49.686,2511.610,7.060,6.580,True
3,19.555,6.240,0.061,137.974,4.355,73.293,29.584,51.439,0.000,0.701,0.001,256.440,49.686,2511.610,6.580,7.940,True
4,20.745,10.021,0.240,97.693,3.155,98.470,49.904,74.187,0.000,0.701,0.001,256.440,49.686,2511.610,7.940,10.890,True


## 6. Pre-training vs fine-tuning split

In [15]:
pretrain_X = data['pretrain_X']
print('Pre-training  (1985-05-01 to 2020-04-01):')
print(f'  pretrain_X : {pretrain_X.shape[0]} rows x {pretrain_X.shape[1]} features')
print()
print('Fine-tuning   (1985-05-01 to 2021-04-14):')
print(f'  finetune_X : {X.shape[0]} rows x {X.shape[1]} features')
print()
print('The fine-tune period extends pre-training by one more year (2020-2021).')
print('It also includes observed stream temperature targets where available.')

Pre-training  (1985-05-01 to 2020-04-01):
  pretrain_X : 12754 rows x 15 features

Fine-tuning   (1985-05-01 to 2021-04-14):
  finetune_X : 13132 rows x 15 features

The fine-tune period extends pre-training by one more year (2020-2021).
It also includes observed stream temperature targets where available.


## 7. Forecast ensemble keys

During the forecast period (Apr–Sep 2021) there are 31 weather ensemble members (E0–E30).  
Each represents a different plausible weather trajectory from the NOAA GEFS model.  
You can use these to produce probabilistic forecasts and quantify uncertainty.

In [16]:
ensemble_keys = [k for k in data.files if k.startswith('forecast_E') and k.endswith('_X')]
print(f'Number of forecast ensemble members: {len(ensemble_keys)}')
print(f'Shape of one ensemble X: {data[ensemble_keys[0]].shape}')
print()
print('Forecast metadata:')
for k in ['forecast_dates', 'forecast_infill_mask', 'forecast_pb_temp']:
    if k in data.files:
        v = data[k]
        print(f'  {k:<25s} shape={v.shape}  dtype={v.dtype}')

Number of forecast ensemble members: 31
Shape of one ensemble X: (708, 15)

Forecast metadata:
  forecast_dates            shape=(708,)  dtype=datetime64[ns]
  forecast_infill_mask      shape=(708,)  dtype=bool
  forecast_pb_temp          shape=(708,)  dtype=float64


## 8. Load all 5 sites and compare

Quick sanity check — confirm shapes are consistent across sites.

In [18]:
rows = []
for sid in SITE_IDS:
    d = np.load(f'{DATA_DIR}/{sid}.npz', allow_pickle=True)
    rows.append({
        'site_id': sid,
        'finetune_rows': d['finetune_X'].shape[0],
        'n_features': d['finetune_X'].shape[1],
        'observed_targets': (~d['finetune_Y_mask'].astype(bool)).sum(),
        'infilled_targets': d['finetune_Y_mask'].astype(bool).sum(),
        'mean_stream_temp_C': round(float(d['finetune_Y'].mean()), 2),
    })
pd.DataFrame(rows)

,site_id,finetune_rows,n_features,observed_targets,infilled_targets,mean_stream_temp_C
0,1573,13132,15,9974,3158,10.270
1,1571,13132,15,8926,4206,9.410
2,1565,13132,15,12809,323,9.710
3,1450,13132,15,12641,491,10.250
4,1641,13132,15,10098,3034,10.510
